# GEMS G13 — GeoDiscovery research engine

One restart-safe notebook from the raw 19-band GeoDAWN raster to geological interpretation, physics features, spatially honest benchmarks, fault-network reconstruction and a validated DrivenData GeoTIFF.

**How to use it**
1. Upload `g13_bundle.zip` to `MyDrive/GEMS_2/` (needed until the G13 code is on `main`).
2. Set `MODE` in the configuration cell. Start with `FAST_BASELINE` (CPU is enough): it writes `submission_g13_baseline.tif` within minutes.
3. Run every cell top to bottom. After a disconnect, run the bootstrap and engine cells again and continue: verified stages are reused, not recomputed.

| Mode | Stages |
|---|---|
| `FAST_BASELINE` | inventory → splits → raw model → S0 submission → DTI tournament → submission → report |
| `FEATURE_DISCOVERY` | + physics features, quality, geological EDA, regimes, S1 physics submission |
| `FULL_RESEARCH` | + ablations A0–A11, incomplete-label learning, optional U-Net, fusion / mixture of experts, topology, failure modes |
| `FINAL_SUBMISSION` | reuses saved models from Drive → tournament → validated champion GeoTIFF |

**Honesty rules built into the code:** labels never enter a feature; TEST geography is used once; every offline DTI is a *proxy* against withheld **known** faults, never the official score; no leaderboard number appears unless you type it in yourself (section 41).

## 01. Configuration and execution modes

In [ ]:
MODE = 'FAST_BASELINE'          # FAST_BASELINE | FEATURE_DISCOVERY | FULL_RESEARCH | FINAL_SUBMISSION
DRIVE_ROOT = '/content/drive/MyDrive/GEMS_2'
WORK_DIR = '/content/gems_g13_work'                       # fast local disk (features live here)
OUT_DIR = DRIVE_ROOT + '/outputs/G13_GEODISCOVERY'        # persistent mirror on Drive
REPO = 'mybenkhadda/GEMS---drivendata'
REF = 'main'
BUNDLE = DRIVE_ROOT + '/g13_bundle.zip'                   # G13 code overlay (skipped if main already has gems_bench/g13)

# Optional overrides (any G13Config key; a typo raises with the closest valid key)
OVERRIDES = dict(
    raw_path='', known_path='', template_path='',          # empty = discovered by content under DRIVE_ROOT
    dem_dir='',                                            # optional 100 m DEM-derived rasters on the template grid
    run_deep=False,                                        # tier-2 U-Net: set True on a GPU runtime (FULL_RESEARCH)
    # n_estimators=400, area_fracs=(0.005, 0.01, 0.02, 0.03, 0.05, 0.08, 0.12),
)
FORCE = ()                       # e.g. ('dti_opt', 'submission') to recompute stages

## 02. GitHub and environment initialization
Mounts Drive, clones `main`, overlays the G13 bundle if `gems_bench/g13` is not on `main` yet, installs only what Colab lacks (no PyTorch / CUDA reinstall), records the commit.

In [ ]:
import os, sys, subprocess, zipfile, importlib
from google.colab import drive
drive.mount('/content/drive')
CODE = '/content/GEMS'
if not os.path.exists(CODE + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '-b', REF, f'https://github.com/{REPO}.git', CODE], check=True)
else:
    subprocess.run(['git', '-C', CODE, 'pull', '-q', '--ff-only'], check=False)
commit = subprocess.run(['git', '-C', CODE, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
if not os.path.exists(CODE + '/gems_bench/g13/engine.py'):
    assert os.path.exists(BUNDLE), f'upload g13_bundle.zip to {BUNDLE}'
    with zipfile.ZipFile(BUNDLE) as z:
        z.extractall(CODE)
    print('G13 overlay extracted from', BUNDLE)
need = [p for p in ('xgboost', 'lightgbm', 'rasterio', 'skimage') if importlib.util.find_spec(p) is None]
if need:
    pip_names = {'skimage': 'scikit-image'}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + [pip_names.get(p, p) for p in need], check=True)
if CODE not in sys.path:
    sys.path.insert(0, CODE)
os.chdir(CODE)
import warnings; warnings.filterwarnings('ignore')
print('commit', commit)

In [ ]:
import json, numpy as np, pandas as pd
from pathlib import Path
from IPython.display import display, Image, Markdown
from gems_bench.g13 import G13_VERSION
from gems_bench.g13 import contracts as C
from gems_bench.g13.config import G13Config
from gems_bench.g13.engine import Engine

cfg = G13Config.from_mode(MODE, drive_root=DRIVE_ROOT, work_dir=WORK_DIR, out_dir=OUT_DIR,
                          force_stages=tuple(FORCE), **OVERRIDES)
eng = Engine(cfg)
W = Path(WORK_DIR)

def run(stage):
    """Run a stage only if it belongs to the selected mode (verified stages are reused)."""
    if stage not in cfg.active_stages():
        print(f'[{stage}] not part of mode {MODE}: skipped'); return None
    return eng.run_stage(stage)

def tbl(rel, n=20):
    p = W / rel
    if p.exists():
        df = pd.read_csv(p); display(df.head(n)); return df
    print('not available:', rel)

def fig(*names):
    idx = W / 'figures' / 'figure_index.csv'
    if not idx.exists(): return
    fi = pd.read_csv(idx).set_index('name')
    for n in names:
        if n in fi.index:
            display(Markdown(f"**{n}** — {fi.loc[n, 'takeaway']}")); display(Image(str(W / fi.loc[n, 'file'])))

print('G13', G13_VERSION, '| mode', MODE, '| stages:', cfg.active_stages())
display(pd.Series(C.hardware_report()))
eng.status()

## 03–04. Input discovery, integrity checks and geophysical inventory
**Motivation:** nothing downstream is trustworthy without verified geometry. **Method:** the three official files are found by *content* (band descriptions, single integer band, same grid); CRS EPSG:32611, 100 m pixels and identical transforms are enforced (no silent reprojection). The raw cube is cached as a local memmap. **Outputs:** band inventory (coverage, quantiles, robust ranges, physical meaning, common-source flags), coverage matrix, figures. **Limitation:** physical units are not in the raster metadata; interpretations marked `inferred` should be checked against the organizers' data dictionary.

In [ ]:
run('inventory')
print(json.dumps(eng.paths, indent=1))
tbl('inventory/band_inventory.csv', 19)
tbl('inventory/band_interpretation.csv', 19)
fig('inventory_coverage', 'inventory_coverage_matrix', 'valid_by_modality', 'raw_multiband_atlas')

## 18 (run early). Geographic and whole-fault validation design
Geographic folds (20 km blocks, 13 km embargo) via `gems_bench.folds`; one fold is the **untouched TEST** partition; selection is cross-fitted on the CV folds; stacking uses out-of-fold predictions. Discovery splits withhold whole fault *systems* (linked within 1 km) from supervision and score them with retained faults masked **pixel-exactly** and their predictions ignored (`disc_official`, the hidden-test analogue). The audit checks disjointness, embargo distance, withheld-label leakage and fault components crossing boundaries.

In [ ]:
run('splits')
tbl('splits/audit.csv')
fig('validation_design')

## 12–17. Physics-informed feature engineering
Families: **magnetic / gravity** (FFT derivatives, THD, tilt, tilt gradient, analytic signal, upward continuation 0.5–4 km and residuals, normalised source strength, multiscale gradient / coherence / Hessian, monogenic local phase and phase congruency, entropy), **conductivity, depth, terrain** (slope, slope breaks, ridge / valley strength, relief anisotropy, TPI), **directional** (oriented log-Gabor), **structural** (cross-contact asymmetry, orientation persistence, characteristic scale), **cross-modal** (axial alignment, joint edges, discordance, shifted cross-correlation and lag, agreement fraction; common-source pairs flagged), **regional normalisation**, **validity** (boundary distance, local coverage). All label-free, mask-aware, stored as robust-scaled memmaps. In `FAST_BASELINE` only the validity family is built. Expect ~10–30 min of CPU for all families on the full grid.

In [ ]:
run('features')
reg = eng.store.registry
if len(reg):
    display(reg.groupby('family').agg(n=('name', 'size'), mb=('bytes', lambda b: b.sum() / 2**20), seconds=('seconds', 'sum')))
fig('feature_atlas')

## 18–19. Feature quality, redundancy and the geological feature registry
Coverage, finite fraction, variance, nodata-boundary dependence, smoothing sensitivity, train/validation shift (KS), redundancy clusters (|Spearman| ≥ 0.98), runtime and storage. Rejections carry an explicit reason; raw bands are never rejected.

In [ ]:
run('quality')
q = tbl('quality/feature_quality.csv', 10)
if q is not None:
    print('kept', int(q.kept.sum()), 'of', len(q)); display(q.reject_reason.value_counts())

## 05–11. Geological EDA: multiscale statistics, fault-conditioned contrasts, cross-fault profiles, directional analysis
Computed on the CV region only (TEST untouched). Contrasts compare the fault corridor with **several control populations** (random, matched 1–3 km, high-gradient, ridge / drainage, magnetic and gravity lineaments without mapped faults, nodata boundaries), with Cliff's delta, spatial-block bootstrap intervals, overlap, mutual information vs a permutation baseline, and fold sign stability. A feature is `discriminative` only if it survives every control; `confounded` if it only separates faults from flat background. Unmapped pixels are unlabeled, not fault-free. Profiles: strike-normal transects (±5 km) clustered into response families.

In [ ]:
run('eda')
s = tbl('eda/discriminative_summary.csv', 25)
if s is not None: display(s.verdict.value_counts())
tbl('eda/scale_space_auc.csv', 40); tbl('eda/correlation_lengths.csv')
fig('scale_space_auc', 'directional_variograms', 'radial_power_spectra', 'fault_conditioned_contrasts',
    'cross_fault_profiles', 'orientation_roses')

## 20–21. Geological regimes and fault archetypes
Clusters of geophysical signatures (never coordinates), fitted on CV training pixels: MiniBatch K-Means across k with seed-stability (ARI), silhouette and GMM BIC; Bayesian GMM and HDBSCAN for comparison. The k rule is fixed in advance. Archetypes are empirical signal categories of mapped-fault pixels (joint magnetic–gravity, strong magnetic, strong gravity, terrain-visible, buried without terrain, weak persistent, weak) with model recall per archetype once a model exists.

In [ ]:
run('regimes')
tbl('regimes/regime_selection.csv'); tbl('regimes/regime_atlas.csv'); tbl('regimes/archetypes.csv')
fig('geological_regimes')

## 25. Raw baseline and **early submission (milestone S0)**
Tier-1 XGBoost (LightGBM too in research modes) on the raw bands: out-of-fold maps on CV folds, one map per discovery split (withheld systems removed, ignore ring weight 0), and a submission model trained on the whole CV region (TEST excluded). `submission_g13_baseline.tif` is written and validated **here**, before any deep learning, clustering or topology. If physics features exist, the physics model and `submission_g13_physics.tif` (S1) follow.

In [ ]:
run('baseline')
board = tbl('tables/scoreboard.csv')
for f in ('submission_g13_baseline', 'submission_g13_physics'):
    p = W / 'submissions' / f'{f}.json'
    if p.exists():
        v = json.loads(p.read_text())['validation']
        print(f"{f}.tif  valid={v['valid']}  failed={v['checks_failed']}  positive_px={v['positive_px']}  Drive: {OUT_DIR}/submissions/{f}.tif")

## 26–27. Feature-family ablations, interactions and compact selection
A0 raw → A1–A8 single families → A9 all → A10 compact (importance + quality) → A11 complementary families; targeted interactions (magnetic × gravity, magnetic × terrain, gravity × depth, conductivity × terrain, structural × cross-modal, structural × directional) with interaction = DTI(A+B) − DTI(A) − DTI(B) + DTI(raw). Identical splits, sampling and models. Paired spatial-block bootstrap against raw-only with Holm correction. This is the most expensive CPU stage (~7 fits per experiment); a GPU runtime makes XGBoost much faster.

In [ ]:
run('ablation')
tbl('ablation/ablation_rankings.csv', 30); tbl('ablation/paired_vs_raw.csv', 30); tbl('ablation/interactions.csv')
fig('scoreboard_disc_proxy_dti', 'scoreboard_geo_proxy_dti')

## 30. Incomplete labels: positive-unlabeled and hard-negative learning
Same features and model, four sampling strategies: standard, distance-stratified, soft negative weights rising with distance, and hard negatives from label-free confounders (ridges, drainages, strong potential-field edges, nodata boundaries) at weight 0.5. Compared with paired block bootstrap under identical validation.

In [ ]:
run('pu_learning')
tbl('ablation/pu_vs_standard.csv')

## 29. Optional deep learning (tier 2/3): multitask residual U-Net
Heads: buffered fault probability (BCE + soft Tversky α=0.2/β=0.8), centre line, axial orientation, distance proxy, junctions; dilated bottleneck. Training tiles lie entirely inside the unit's training mask; Hann-blended inference. Compared with a single-head variant; the simpler model is kept unless multitask wins on held-out DTI. Runs only with `run_deep=True` and torch available (use a GPU runtime).

In [ ]:
run('deep')
b = tbl('tables/scoreboard.csv', 50)
if b is not None: display(b[b.model.str.startswith('DL_')])

## 23 & 31. Model disagreement and adaptive expert fusion
Magnetic, gravity and terrain experts are trained, then the best complementary experts are fused as per-zone ranks: rank mean, cross-fitted fixed weights, and a LightGBM gate on [expert ranks, regime, DEM / boundary reliability, coherence, cross-modal agreement, disagreement] trained on out-of-fold predictions only. The gate is kept only if it beats both simpler fusions. Disagreement regions are listed as candidates for review.

In [ ]:
run('fusion')
tbl('fusion/fusion_scoreboard.csv'); tbl('fusion/disagreement_candidates.csv')

## 32–35. Full-grid prediction, candidate lineaments, fault-network topology and discovery atlas
The champion's full map → top-a mask → small-component removal and orientation-aware gap linking → skeleton graph (segments, junctions, geodesic length, tortuosity, curvature) → graph confidence propagation → classification relative to mapped faults (`mapped_trace`, `known_extension`, `parallel`, `intersecting`, `unverified_network`, `isolated`) with evidence columns. **A candidate is a prediction, not a discovered fault.** Exported as CSV and GeoJSON (EPSG:32611 centroids).

In [ ]:
run('topology')
tbl('topology/network_summary.csv'); tbl('topology/candidates.csv', 25)
fig('discovery_atlas')

## 36–37. DTI sensitivity, postprocessing tournament and once-only TEST evaluation
Synthetic experiments show how DTI responds to width, displacement, background, gamma, false-positive length and recall. The tournament scores raw probabilities, output transforms, binary area budgets, crest (NMS) lines, small-component removal, gap linking, regime-balanced budgets, soft outputs and a known-fault ring on every validation unit; each unit's choice is made on the other units, and the full search table is saved. The selected (model, representation) is then evaluated **once** on the untouched TEST fold.

In [ ]:
run('dti_opt')
p = W / 'dti_opt' / 'selection.json'
if p.exists():
    sel = json.loads(p.read_text())
    print('champion model:', sel['champion_model'], '| representation:', sel['key'])
    print('cross-fitted:', {t: round(v['dti_crossfit'], 4) for t, v in sel['crossfit'].items()})
    print('TEST proxy (once):', round(sel['test']['test_proxy_dti'], 4), '-', sel['test']['note'])
tbl('dti_opt/representation_families.csv', 30); tbl('dti_opt/width_displacement.csv')
fig('dti_synthetic_sensitivity', 'dti_vs_area')

## 22 (on held-out predictions). Failure modes and error taxonomy
Missed faults (all / weak archetype), broken continuity, displacement, thick ribbons, isolated noise, false positives on ridges, drainages, nodata boundaries and near known faults, spurious junctions. Each category names the tournament operation or experiment that tests its remedy.

In [ ]:
run('failure_modes')
tbl('failure_modes/taxonomy.csv', 30)
fig('failure_mode_panels')

## 38–40. Submission candidates, final GeoTIFF validation and export
Candidates **A** (safe raw baseline), **B** (best physics model), **C** (discovery champion with the tournament representation). Each is written on the official template grid (float32, [0, 1], NaN outside the template, 0 on pixel-exact known faults, no buffer) and validated (geometry, dtype, NaN pattern, range, non-degenerate, checksum, reopen). The champion is chosen by evidence and copied to `submission_g13_champion.tif` and `predicted_faults.tif`.

In [ ]:
run('submission')
tbl('submissions/candidate_comparison.csv')
p = W / 'submissions' / 'champion.json'
if p.exists():
    ch = json.loads(p.read_text())
    display(Markdown(f"""**Champion:** `{ch['file']}` (candidate {ch['chosen_candidate']}, model `{ch['model']}`)  
**Selection:** {ch['selection_rule']}  
**Files on Drive:** `{OUT_DIR}/submissions/predicted_faults.tif` (identical copy: `submission_g13_champion.tif`)  
**Download:** Drive web UI, or `from google.colab import files; files.download('{WORK_DIR}/submissions/predicted_faults.tif')`  
**Submit:** https://www.drivendata.org/competitions/306/competition-doe-gems/ → *Submissions* → upload the file. Nothing is uploaded automatically.  
**Leaderboard score:** {ch['leaderboard_score']}"""))

## 41. Leaderboard feedback (manual)
After the platform shows a score, type it in here. It is appended to `submission_history.csv` next to the offline proxies so the proxy–leaderboard gap can be studied. Do not use repeated uploads as a parameter search (3 scored submissions per rolling 7 days).

In [ ]:
SUBMITTED_FILE = ''     # e.g. 'predicted_faults.tif'
PUBLIC_SCORE = None     # the number shown by DrivenData
if SUBMITTED_FILE and PUBLIC_SCORE is not None:
    display(eng.log_leaderboard(SUBMITTED_FILE, float(PUBLIC_SCORE), notes=MODE))
else:
    print('no leaderboard score logged')

## 42. Scientific report and next experiments
Markdown + HTML report with the 15 required sections, findings judged from measured effects (`supported` needs a positive effect, a block-bootstrap interval above zero and a positive sign in every zone), figure index and runtime table.

In [ ]:
run('report')
eng.ws.persist()
display(eng.status())
tbl('reports/findings.csv')
display(Markdown((W / 'reports' / 'report.md').read_text()[:6000]))
print('full report:', OUT_DIR + '/reports/report.html')